# Install & Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from snowflake.snowpark.context import get_active_session

# Configuration for plots
plt.style.use('ggplot')
sns.set_palette("colorblind")

# Data Ingestion

In [ ]:
session = get_active_session()

def load_csv_from_stage(stage_path: str, file_name: str) -> pd.DataFrame:
    """
    Loads a CSV file from a Snowflake stage directly into a Pandas DataFrame.
    """
    download_path = f"/tmp/{file_name}"
    session.file.get(f"{stage_path}/{file_name}", "/tmp/")
    return pd.read_csv(download_path)

In [ ]:
data_stage = "@EXTERNAL_DATA_STAGE" 

raw_water_quality = load_csv_from_stage(data_stage, "water_quality_training_dataset.csv")
landsat_features = load_csv_from_stage(data_stage, "landsat_features_training.csv")
terraclimate_features = load_csv_from_stage(data_stage, "terraclimate_features_training.csv")
external_features = load_csv_from_stage(data_stage, "external_features_training.csv")

print(f"Base records: {len(raw_water_quality)}")

In [ ]:
def merge_feature_datasets(base_df: pd.DataFrame, feature_dataframes: list, merge_keys: list) -> pd.DataFrame:
    """
    Iteratively merges multiple feature dataframes into the base dataframe.
    """
    merged_df = base_df.copy()
    for feature_df in feature_dataframes:
        merged_df = pd.merge(
            merged_df, 
            feature_df, 
            on=merge_keys, 
            how='left'
        )
    return merged_df

merge_columns = ['Latitude', 'Longitude', 'Sample Date']
datasets_to_merge = [landsat_features, terraclimate_features, external_features]

master_dataset = merge_feature_datasets(raw_water_quality, datasets_to_merge, merge_columns)

# Drop duplicate columns if any were created during merge
master_dataset = master_dataset.loc[:, ~master_dataset.columns.duplicated()]

# Sanity Check's

In [ ]:
def analyze_missing_values(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates the count and percentage of missing values for each column.
    """
    missing_counts = dataframe.isnull().sum()
    missing_percentages = (missing_counts / len(dataframe)) * 100
    
    missing_summary = pd.DataFrame({
        'missing_count': missing_counts,
        'missing_percentage': missing_percentages
    })
    
    return missing_summary[missing_summary['missing_count'] > 0].sort_values(by='missing_percentage', ascending=False)

missing_report = analyze_missing_values(master_dataset)
display(missing_report)

In [ ]:
target_metrics = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for index, target in enumerate(target_metrics):
    sns.histplot(master_dataset[target].dropna(), kde=True, ax=axes[index])
    axes[index].set_title(f"Distribution of {target}")

plt.tight_layout()
plt.show()

In [ ]:
def plot_target_correlations(dataframe: pd.DataFrame, targets: list, top_n: int = 15):
    """
    Plots the top N correlated features for each target variable.
    """
    numeric_df = dataframe.select_dtypes(include=[np.number])
    correlation_matrix = numeric_df.corr()
    
    for target in targets:
        if target in correlation_matrix.columns:
            correlations = correlation_matrix[target].drop(targets).abs().sort_values(ascending=False)
            top_features = correlations.head(top_n)
            
            plt.figure(figsize=(10, 6))
            sns.barplot(x=top_features.values, y=top_features.index)
            plt.title(f"Top {top_n} Features Correlated with {target}")
            plt.xlabel("Absolute Pearson Correlation")
            plt.show()

plot_target_correlations(master_dataset, target_metrics)

In [ ]:
master_dataset.to_csv("/tmp/master_training_dataset.csv", index=False)
session.file.put("/tmp/master_training_dataset.csv", "@EXTERNAL_DATA_STAGE/", auto_compress=False, overwrite=True)
print("Master dataset saved successfully.")